In [1]:
%load_ext autoreload
%autoreload 2

from tasks.autoencoder import AETask
import os
import torch
from tqdm import tqdm
import einx
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import Pooling, Transformer, Normalize
from transformers import AutoModel, AutoTokenizer, T5TokenizerFast, AutoModelForCausalLM, T5Tokenizer, GPT2LMHeadModel
from datasets import load_dataset, load_from_disk
import re
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from data.datasets import WikipediaDataset, WikipediaDatasetConfig
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, models
import wandb
from tasks.dlclm import DLCLMTask, InfoLabel
from mauve import compute_mauve, get_features_from_input
from model.encoder import HSEMHead, HSEMHeadConfig, SEMHeadConfig, SEMHead
import os
from data.datasets import FineWebDataset
from transformers import GenerationConfig

/home/l/leog/links/scratch/sentence_diffusion/venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/l/leog/links/scratch/sentence_diffusion/venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assign

In [2]:
task = DLCLMTask.load_from_checkpoint(
    os.path.join(
        os.environ["LOG_DIR"],
        "checkpoints/",
        "u3wlarr4",
        "last.ckpt",
    ),
    strict=False,
    map_location=torch.device("cpu"),
)
task.setup()

FileNotFoundError: [Errno 2] No such file or directory: '/scratch/l/leog/sentence_diffusion/logs/checkpoints/u3wlarr4/last.ckpt'

In [3]:
task = task.to(device=torch.device("cuda"), dtype=torch.bfloat16)

In [18]:
batch = next(iter(task.val_dataloader()))

In [19]:
prompt = [
    batch["input_ids_dec"][i][
        (batch["info_mask_dec"][i] == InfoLabel.DLC.value)
        + (batch["info_mask_dec"][i] == InfoLabel.PROMPT.value)
    ].tolist()
    for i in [0,0,0,0,0,0,0,0]
]

In [26]:
torch.ones((10,10), dtype=torch.long).cumsum(-1) - 1 

tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]])

In [20]:
prompt_str = task.decoder.tokenizer.batch_decode(prompt)
prompt_str[0]

'One in five adults will be diagnosed with a mental illness this year. The numbers are starting, but they don’t have to be. Mental health is a cause you can support with every stride. Hope Network’s One in Five Marathon Relay takes place on Friday, June 8 at twilight<|think|>'

In [22]:
continuation = task.decoder.generate(
        prompt=prompt,
        max_length=task.dataset.max_length,
        gen_kwargs={'do_sample': True, "max_new_tokens": 96}
    )
continuation_str = task.decoder.tokenizer.batch_decode(
    continuation,
    skip_special_tokens=True
)

In [23]:
continuation_str

['. If you are interested in participating as a race participant or a friend, click here to join the One in Five Relay fundraising effort.\nPawtucket Running Group runs in May, 2017\nThe Paws Up Pawtucket Running group is located in the city of Pawtucket, Rhode Island. To find out more, visit our web site at www.owspawtucketrunning.org.\nView the 2021 Paws Up Pawtucket Running Times',
 '. Join us, and help us all know that our loved ones are being supported through treatment. Learn more\nOne in 5 takes place on World Mental Health Day, June 8, 2019, at darkmountain.com.\nFor more information, go to www.horizonshealth.us\nFor more help for mental illness, visit https://horizonshealing.org\nFor more information, go to http://www.horizonshope.org\n© 2024',
 '. Help us raise awareness and find a way to support our community of believers. Proceeds benefit Hope Network training and therapy services.\nThe race starts at sunset on June 8.\nWhere and when: Hope Network’s North American Mental H

In [15]:
batch_ = task.decoder.tokenizer.pad(
    {"input_ids": [p + [task.decoder.tokenizer.bos_token_id] for p in prompt]},
    padding=True,
    padding_side="left",
    return_tensors="pt",
).to('cuda')
gen_cfg = {
    "max_new_tokens": 96,  # 32-token DLC + closing <|bos|>
    "do_sample": False,
    "top_p": 1.0,
    "top_k": 50,
    "num_beams": 1,
    "temperature": 1.0,
    "return_dict_in_generate": True,
    "pad_token_id": task.decoder.tokenizer.pad_token_id,
    "eos_token_id": task.decoder.tokenizer.eos_token_id,
    "use_cache": True,
}

task.decoder.tokenizer.batch_decode(task.decoder.backbone.generate(
    **batch_,
    generation_config=GenerationConfig(**gen_cfg),
).sequences)[0]

'One in five adults will be diagnosed with a mental illness this year. The numbers are starting, but they don’t have to be. Mental health is a cause<|think|><|endoftext|> of so much suffering and so much loss.\nThe Mental Health Awareness Week is on March 5-7, 2014. The Mental Health Awareness Week is a week-long celebration of mental health.\nThe Mental Health Awareness Week is on March 5-7, 2014.\nThe Mental Health Awareness Week is on March 5-7, 2014.\nFor more information, visit the Mental Health Awareness Week website.\nFor more information, visit the Mental Health Awareness Week website.\n'

In [17]:
tok = task.decoder.tokenizer
model = task.decoder.backbone

In [21]:
prompt = [p + [tok.bos_token_id] for p in prompt]

In [59]:
batch = tok.pad({"input_ids": prompt}, padding=True, padding_side='left', return_tensors='pt').to('cuda')

In [10]:
model.config.pad_token_id = tok.pad_token_id
model.config.vocab_size = len(model.transformer.wte.weight)

In [26]:
batch["input_ids"][0]

tensor([ 3198,   287,  1936,  6490,   481,   307, 14641,   351,   257,  5110,
         8526,   428,   614,    13,   383,  3146,   389,  3599,    11,   475,
          484,   836,   447,   247,    83,   423,   284,   307,    13, 21235,
         1535,   318,   257,  2728,   345,   460,  1104,   351, 50258, 50299,
        50288, 50287, 50287, 50312, 50288, 50302, 50296, 50294, 50274, 50262,
        50291, 50322, 50315, 50315, 50311, 50302, 50314, 50262, 50305, 50321,
        50262, 50310, 50311, 50266, 50315, 50263, 50261, 50312, 50301, 50265,
        50260, 50320, 50276, 50303, 50307, 50302, 50321, 50285, 50260, 50283,
        50319, 50288, 50284, 50294, 50311, 50302, 50269, 50272, 50302, 50318,
        50289, 50263, 50259, 50305, 50304, 50297, 50277, 50284, 50307, 50300,
        50272, 50297, 50297, 50256], device='cuda:0')

In [27]:
batch["attention_mask"][0]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1], device='cuda:0')

In [63]:
gen_cfg = {
    "max_new_tokens": 96,  # 32-token DLC + closing <|bos|>
    "do_sample": False,
    "top_p": 1.0,
    "top_k": 50,
    "num_beams": 1,
    "temperature": 1.0,
    "return_dict_in_generate": True,
    "pad_token_id": tok.pad_token_id,
    "eos_token_id": tok.eos_token_id,
    "use_cache": True,
    "return_attention": True
}
continuation_bad = model.generate(**batch,generation_config=GenerationConfig(**gen_cfg))
continuation = model.generate(input_ids = batch["input_ids"][0][batch["attention_mask"][0].bool()][None],generation_config=GenerationConfig(**gen_cfg))

print("without padding: ", tok.decode(continuation.sequences[0]))
print("with padding: ", tok.decode(continuation_bad.sequences[0]))

without padding:  One in five adults will be diagnosed with a mental illness this year. The numbers are starting, but they don’t have to be. Mental health is a cause you can support with every stride. Hope Network’s One in Five Marathon Relay takes place on Friday, June 8<|think|><|endoftext|>, at 7:30 a.m. and is a great way to support mental health awareness and awareness of mental illness.
One in Five Marathon Relay
June 8, 2017
Hope Network, Inc.
For more information, please visit www.HopeNetwork.org.
For more information, please call (800) 852-5100.
Hope Network, Inc.
P.O. Box 515
Canton, OH 44142
Copyright
with padding:  One in five adults will be diagnosed with a mental illness this year. The numbers are starting, but they don’t have to be. Mental health is a cause you can support with every stride. Hope Network’s One in Five Marathon Relay takes place on Friday, June 8<|think|><|endoftext|>, at 7:30 a.m. and is a great way to support mental health awareness and awareness of men

In [64]:
continuation

GenerateDecoderOnlyOutput(sequences=tensor([[ 3198,   287,  1936,  6490,   481,   307, 14641,   351,   257,  5110,
          8526,   428,   614,    13,   383,  3146,   389,  3599,    11,   475,
           484,   836,   447,   247,    83,   423,   284,   307,    13, 21235,
          1535,   318,   257,  2728,   345,   460,  1104,   351,   790, 33769,
            13, 13408,  7311,   447,   247,    82,  1881,   287, 10579, 24828,
          4718,   323,  2753,  1295,   319,  3217,    11,  2795,   807, 50258,
         50280, 50288, 50275, 50281, 50278, 50288, 50314, 50284, 50298, 50274,
         50305, 50267, 50322, 50286, 50318, 50305, 50305, 50314, 50262, 50297,
         50306, 50262, 50287, 50278, 50280, 50315, 50296, 50261, 50290, 50301,
         50265, 50288, 50320, 50276, 50303, 50293, 50293, 50307, 50285, 50260,
         50283, 50320, 50281, 50284, 50294, 50311, 50262, 50269, 50287, 50296,
         50290, 50305, 50283, 50259, 50312, 50294, 50266, 50303, 50284, 50307,
         50298, 

In [11]:
input_ids = batch["input_ids"][0]
mask = batch["attention_mask"][0]

In [12]:
position_ids = (mask.cumsum(-1) - 1).clamp(min=0)
position_ids = position_ids * mask  # 0 on pads, 0..L-1 on tokens

In [25]:
model.config.use_cache = False

In [42]:
model = model.to(dtype=torch.bfloat16)

In [50]:
model.set_attn_implementation("sdpa")

In [51]:
print(input_ids[mask.bool()])
print(position_ids[mask.bool()])
out1 = model(input_ids=input_ids[None], attention_mask=mask[None], position_ids=position_ids[None])
out1.logits[0,-1].argmax()

tensor([ 3198,   287,  1936,  6490,   481,   307, 14641,   351,   257,  5110,
         8526,   428,   614,    13,   383,  3146,   389,  3599,    11,   475,
          484,   836,   447,   247,    83,   423,   284,   307,    13, 21235,
         1535,   318,   257, 50258, 50299, 50288, 50287, 50287, 50312, 50288,
        50302, 50280, 50294, 50274, 50319, 50267, 50322, 50315, 50315, 50311,
        50302, 50296, 50262, 50305, 50321, 50291, 50268, 50295, 50266, 50315,
        50263, 50299, 50301, 50301, 50265, 50285, 50320, 50276, 50303, 50307,
        50302, 50319, 50285, 50309, 50283, 50269, 50288, 50284, 50294, 50311,
        50302, 50269, 50311, 50302, 50318, 50289, 50283, 50259, 50305, 50304,
        50318, 50277, 50284, 50307, 50279, 50306, 50297, 50297, 50256],
       device='cuda:0')
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44

tensor(2219, device='cuda:0')

In [52]:
print(input_ids[mask.bool()][None])
out2 = model(input_ids=input_ids[mask.bool()][None])
out2.logits[0,-1].argmax()

tensor([[ 3198,   287,  1936,  6490,   481,   307, 14641,   351,   257,  5110,
          8526,   428,   614,    13,   383,  3146,   389,  3599,    11,   475,
           484,   836,   447,   247,    83,   423,   284,   307,    13, 21235,
          1535,   318,   257, 50258, 50299, 50288, 50287, 50287, 50312, 50288,
         50302, 50280, 50294, 50274, 50319, 50267, 50322, 50315, 50315, 50311,
         50302, 50296, 50262, 50305, 50321, 50291, 50268, 50295, 50266, 50315,
         50263, 50299, 50301, 50301, 50265, 50285, 50320, 50276, 50303, 50307,
         50302, 50319, 50285, 50309, 50283, 50269, 50288, 50284, 50294, 50311,
         50302, 50269, 50311, 50302, 50318, 50289, 50283, 50259, 50305, 50304,
         50318, 50277, 50284, 50307, 50279, 50306, 50297, 50297, 50256]],
       device='cuda:0')


tensor(2219, device='cuda:0')

In [21]:
print(input_ids[mask.bool()][None])
out3 = model(input_ids=input_ids[None])
out3.logits[0,-1].argmax()

tensor([[ 3198,   287,  1936,  6490,   481,   307, 14641,   351,   257,  5110,
          8526,   428,   614,    13,   383,  3146,   389,  3599,    11,   475,
           484,   836,   447,   247,    83,   423,   284,   307,    13, 21235,
          1535,   318,   257, 50258, 50299, 50288, 50287, 50287, 50312, 50288,
         50302, 50280, 50294, 50274, 50319, 50267, 50322, 50315, 50315, 50311,
         50302, 50296, 50262, 50305, 50321, 50291, 50268, 50295, 50266, 50315,
         50263, 50299, 50301, 50301, 50265, 50285, 50320, 50276, 50303, 50307,
         50302, 50319, 50285, 50309, 50283, 50269, 50288, 50284, 50294, 50311,
         50302, 50269, 50311, 50302, 50318, 50289, 50283, 50259, 50305, 50304,
         50318, 50277, 50284, 50307, 50279, 50306, 50297, 50297, 50256]],
       device='cuda:0')


tensor(2219, device='cuda:0')

In [199]:
model.config.use_cache=False

In [205]:
out1, out2

(CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[-0.7578,  2.3125, -2.3281,  ..., -3.4219, -1.6719, -3.9375],
          [-0.7578,  2.3125, -2.3281,  ..., -3.4219, -1.6719, -3.9375],
          [-0.7578,  2.3125, -2.3281,  ..., -3.4219, -1.6719, -3.9375],
          ...,
          [ 4.7812,  1.0234,  2.5781,  ..., -0.3262,  1.0938,  0.1367],
          [ 4.5938,  0.4551,  2.0938,  ..., -1.0000,  0.3945, -0.5625],
          [ 4.1250,  0.1670,  1.6016,  ..., -1.7891, -0.5117, -1.2578]]],
        device='cuda:0', dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>), past_key_values=None, hidden_states=None, attentions=None, cross_attentions=None),
 CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[-0.5898,  1.7344, -0.5352,  ..., -3.2344, -2.1875, -3.6562],
          [ 1.2578, -0.1396, -3.0469,  ..., -4.3438, -3.9219, -4.7500],
          [ 2.6094,  0.5703, -2.6406,  ..., -4.4688, -3.5625, -4.4688],
          ...,
          [-2.4844, -3.9844, -5.2188,  ..., 14.6250, 1

In [143]:
model.get_input_embeddings().num_embeddings-len(tok)

64

50259

'[PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD]One in five adults will be diagnosed with a mental illness this year. The numbers are starting, but they don’t have to be. Mental health is a cause you can support with every stride. Hope Network’s One<|think|><|endoftext|> (That Day (Posted the Angry With A (This April F (Posted After More\nFriday('

'One in five adults will be diagnosed with a mental illness this year. The numbers are starting, but they don’t have to be. Mental health is a cause you can support with every stride. Hope Network’s One<|think|><|endoftext|> in Five Campaign will raise awareness during February through March.\nTogether, raise awareness, encourage conversation and'

In [38]:
data = FineWebDataset(length_interval=[32, 96])

Resolving data files:   0%|          | 0/824 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/828 [00:00<?, ?it/s]

In [44]:
batch = data.__getitems__([0,1,2,3,4,5,6,7,8,9,10,11,12])

In [207]:
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device='cuda')

# We recommend enabling flash_attention_2 for better acceleration and memory saving,
# together with setting `padding_side` to "left":
# model = SentenceTransformer(
#     "Qwen/Qwen3-Embedding-0.6B",
#     model_kwargs={"attn_implementation": "flash_attention_2", "device_map": "auto"},
#     tokenizer_kwargs={"padding_side": "left"},
# )

In [208]:
#prompt = "Represent the text with high-level features. This can include semantic information, type of text, intent, style. Text: "
#prompt = "Instruct: Represent the text with high-level features. This can include semantic information, type of text, intent, style.\nText: "
prompt = None
query_embeddings = model.encode(batch["input_str"], prompt=prompt)
document_embeddings = model.encode(batch["input_str"], prompt=prompt)
similarity = model.similarity(query_embeddings, document_embeddings)
similarity.fill_diagonal_(-torch.inf);

In [204]:
batch["input_str"][1]

"\nI'm not going to lie to you and say that I loved everything about it (3:00 AM wake up calls being the main offender), but I was consistently surprised to find that even in the tougher times, when we had been blearily working for 18 hours straight, something or someone would come along to pick everyone up.\nFrom the absolute chaos of pre-show preparations,"

In [206]:
batch["input_str"][8]

' making Monday nights on SPEED interesting again. It also helped that the network executives finally relented and let the race review lead the show. It made a big difference.\nThis week, Byrnes, Waltrip and Knaus were all tired from a long California weekend and a three hour time zone shift. Waltrip started slow, but got himself back on-track by suggesting California go to'

In [205]:
torch.softmax(similarity/0.01,-1)[1].argsort(descending=True)

tensor([ 8,  9,  7, 11,  6, 12, 10,  3,  2,  4,  5,  0,  1])

In [210]:
torch.softmax(similarity/0.01,-1)[0].argsort(descending=True)

tensor([ 7, 12,  2, 10,  1,  9, 11,  4,  3,  8,  6,  5,  0])

In [169]:
print("A:" +batch["input_str"][1] + "\nB:" + batch["input_str"][6])

A:
I'm not going to lie to you and say that I loved everything about it (3:00 AM wake up calls being the main offender), but I was consistently surprised to find that even in the tougher times, when we had been blearily working for 18 hours straight, something or someone would come along to pick everyone up.
From the absolute chaos of pre-show preparations,
B:ly got 28 first-place votes versus only 11 for Wagner.
At times like these I can imagine some fans crying about an East Coast bias within the media, but I'm not sure how much I buy that here given the fact that the


In [164]:
# The queries and documents to embed
queries = [
    "Then, I went to the garden to the garden to pick some cherries.",
    "Explain gravity.",
]
documents = [
    "Last, year, diego went to Africa to meet an old friend.",
    "Gravity is a force that attracts two bodies towards each other.",
    "I love going in the forest and foraging mushrooms!",
    "You love going in the forest and foraging mushrooms!",
    "Can you explain the maillard reaction?",
    "Explain why the sky is blue.",
    "Yesterday, I went to the store to buy some things."
]
prompt = None

#prompt = "Instruct: Given some text, retrieve other texts which share some high level attributes. This can include semantic information, type of text, intent, style. \nText:"
prompt = "Represent the text with high-level features. This can include semantic information, type of text, intent, style. Text: "
#prompt = "Instruct: Given some text taken from the internet, retrieve other similar text. Text: "
#prompt = "Instruct: Represent the following passage with its high-level features. This can include semantic information, type of text, intent, style.\nText: "
#prompt = "Instruct: Represent the following text "
query_embeddings = model.encode(queries, prompt=prompt)
document_embeddings = model.encode(documents, prompt=prompt)

In [165]:
similarity = model.similarity(query_embeddings, document_embeddings)
similarity

tensor([[0.8175, 0.8386, 0.9061, 0.8952, 0.8412, 0.8487, 0.8798],
        [0.8261, 0.9366, 0.9186, 0.9336, 0.9719, 0.9821, 0.8365]])

In [166]:
torch.softmax(similarity/0.1,-1)

tensor([[0.0882, 0.1089, 0.2141, 0.1919, 0.1119, 0.1206, 0.1645],
        [0.0509, 0.1537, 0.1284, 0.1492, 0.2189, 0.2424, 0.0565]])

In [153]:
import torch

In [15]:
x = torch.IntTensor([[1,2,3],[4,5,6]])
prompt = [3,3,4,4]

In [22]:
[[] for i in range(10)]

[[], [], [], [], [], [], [], [], [], []]

In [18]:
torch.cat([x,y])

RuntimeError: Tensors must have same number of dimensions: got 2 and 1

In [7]:
[prompt + x]

TypeError: can only concatenate list (not "Tensor") to list